In [2]:
import torch
import random
import torch.nn.functional as F
import numpy as np
from math import *
import matplotlib.pyplot as plt
from torch.autograd import Variable
from IPython import display
import torch.utils.data as Data
import torch.nn as nn
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.ticker import LinearLocator, FormatStrFormatter
from matplotlib import cm
from scipy.linalg import block_diag

from torch.nn.utils import *

In [3]:
def DFT_matrix(N):
    i, j = np.meshgrid(np.arange(N), np.arange(N))
    omega = np.exp( - 2 * pi * 1J / N )
    W = np.power( omega, i * j ) / sqrt(N)
    return np.mat(W)

# ULA / UPA

In [4]:
def freqency_sparse_SV_channel0(Nc,N,sigma_2_alpha,Nt, tau=None, alpha=None, fai1=None):
        # Nc -- Number of subcarriers
        # N -- Number of paths 
        # sigma_2_alph -- Variance of the amplitude of each path 
    d=0.5
    #print(check)
    Tau = Nc  # Maximum delay spread
    if tau is None:
        tau = np.random.rand(1,N)*Tau 

    if isinstance(Nt,int):  # ULA
        #n_r = np.mat(range(Nr)).reshape(Nr,1)
        n_t = np.mat(range(Nt)).reshape(Nt,1)
        if fai1 is None:
            fai1 = np.random.rand(1,N)*2*pi # Random angles of departure
        A_t = np.mat(np.zeros([Nt,N],dtype = complex))
        A_r = np.mat(np.ones([1,N],dtype = complex))
        alpha = np.squeeze(np.zeros([1,N],dtype = complex),0)
        H_f = np.array(np.zeros([1,Nc,Nt],dtype = complex))
        for i in range(N):
            A_t[:,i] = np.exp(-2j*pi*0.5*n_t*np.sin(fai1[0,i]))
            if alpha is None:
                aa = (np.random.randn(1,1)+1j*np.random.randn(1,1))*np.sqrt(sigma_2_alpha/2/N)
                alpha[i] = aa[0,0] # Gaussian-distributed amplitude of the path 

    else: # UPA 
        N_r = Nr
        N_t = Nt[0]*Nt[1]
        fait = np.random.rand(1,N)*2*pi
        fair = np.random.rand(1,N)*2*pi
        theatt = np.random.rand(1,N)*2*pi
        theatr = np.random.rand(1,N)*2*pi
        A_t = np.mat(np.zeros([N_t,N],dtype = complex))
        A_r = np.mat(np.zeros([N_r,N],dtype = complex))
        alpha = np.squeeze(np.zeros([1,N],dtype = complex),0)
        H_f = np.array(np.zeros([N_r,N_t,Nc],dtype = complex))
        n_t1 = np.mat(range(Nt[0])).reshape(Nt[0],1)
        n_t2 = np.mat(range(Nt[1])).reshape(Nt[1],1)
        A_r = np.mat(np.ones([1,N],dtype = complex))
        H_f = np.array(np.zeros([1,Nc,N_t],dtype = complex))
        for i in range(N):
            at1 = np.exp(-2j*pi*0.5*n_t1*np.cos(fait[0,i])*np.sin(theatt[0,i]))
            at2 = np.exp(-2j*pi*0.5*n_t2*np.sin(fait[0,i]))
            A_t[:,i] = np.kron(at1,at2)

            aa = (np.random.randn(1,1)+1j*np.random.randn(1,1))*np.sqrt(sigma_2_alpha/2/N)  
            alpha[i] = aa[0,0]
                
                
    for k in range(Nc):
        P = np.squeeze(alpha*np.exp(-1j*2*pi*tau*k/Nc),0);
        P = np.diag(P)
        H_f[:,k,:] = np.dot(np.dot(A_r,P),A_t.H)
        
    return H_f,A_t,alpha

In [9]:
Nc = 32
sigma_2_alpha = 1
Nt = 8
if isinstance(Nt,int):
    N_t = Nt
else:
    N_t = Nt[0]*Nt[1]
Nr = 1

B = 30
L = 8

SNR_dB = 10
K = 4 # number of users
snr =  10**(SNR_dB/10)/K

N_BATCH_train = 1
N_BATCH_test  = 80
BATCH_SIZE = 1
N_H_train = N_BATCH_train*BATCH_SIZE
N_H_test = N_BATCH_test*BATCH_SIZE

In [ ]:
for N in range(1,9):
    H_torch = torch.zeros([BATCH_SIZE*N_BATCH_train,K,Nc,N_t*2])
    for i in range(N_BATCH_train):
        H = np.zeros([BATCH_SIZE,K,Nc,N_t],dtype=complex) #D0: samples, D1: users, D2: subcarriers, D3: antennas
        for j in range(BATCH_SIZE):
            for k in range(K):
                H_f,A_t,alpha = freqency_sparse_SV_channel0(Nc,N,sigma_2_alpha,Nt)
                H[j,k,:,:] = H_f
    
        H_torch[(i*BATCH_SIZE):((i+1)*BATCH_SIZE),:,:,0:N_t] = torch.from_numpy(np.real(H))
        H_torch[(i*BATCH_SIZE):((i+1)*BATCH_SIZE),:,:,N_t:2*N_t] = torch.from_numpy(np.imag(H))
    #torch.save(H_torch,'data/H_train_UPA'+str(N)+'Lp_1.pt')
    
    H_torch = torch.zeros([BATCH_SIZE*N_BATCH_train,K,Nc,N_t*2])
    for i in range(N_BATCH_train):
        H = np.zeros([BATCH_SIZE,K,Nc,N_t],dtype=complex) #D0: samples, D1: users, D2: subcarriers, D3: antennas
        for j in range(BATCH_SIZE):
            for k in range(K):
                H_f,A_t,alpha = freqency_sparse_SV_channel0(Nc,N,sigma_2_alpha,Nt)
                H[j,k,:,:] = H_f
    
        H_torch[(i*BATCH_SIZE):((i+1)*BATCH_SIZE),:,:,0:N_t] = torch.from_numpy(np.real(H))
        H_torch[(i*BATCH_SIZE):((i+1)*BATCH_SIZE),:,:,N_t:2*N_t] = torch.from_numpy(np.imag(H))
    #torch.save(H_torch,'data/H_train_UPA'+str(N)+'Lp_2.pt')

In [11]:
for N in range(1,9):
    H_torch = torch.zeros([BATCH_SIZE*N_BATCH_test,K,Nc,N_t*2])
    for i in range(N_BATCH_test):
        H = np.zeros([BATCH_SIZE,K,Nc,N_t],dtype=complex) #D0: samples, D1: users, D2: subcarriers, D3: antennas
        for j in range(BATCH_SIZE):
            for k in range(K):
                H_f,A_t,alpha = freqency_sparse_SV_channel0(Nc,N,sigma_2_alpha,Nt)
                H[j,k,:,:] = H_f
    
        H_torch[(i*BATCH_SIZE):((i+1)*BATCH_SIZE),:,:,0:N_t] = torch.from_numpy(np.real(H))
        H_torch[(i*BATCH_SIZE):((i+1)*BATCH_SIZE),:,:,N_t:2*N_t] = torch.from_numpy(np.imag(H))
    #torch.save(H_torch,'data/H_test_UPA'+str(N)+'Lp.pt')

In [13]:

H_torch = torch.zeros([BATCH_SIZE*N_BATCH_train,K,Nc,N_t*2])
for i in range(N_BATCH_train):
    H = np.zeros([BATCH_SIZE,K,Nc,N_t],dtype=complex) #D0: samples, D1: users, D2: subcarriers, D3: antennas
    for j in range(BATCH_SIZE):
        N = np.int(np.random.rand(1)*8)+1
        for k in range(K):
            H_f,A_t,alpha = freqency_sparse_SV_channel0(Nc,N,sigma_2_alpha,Nt)
            H[j,k,:,:] = H_f
    
    H_torch[(i*BATCH_SIZE):((i+1)*BATCH_SIZE),:,:,0:N_t] = torch.from_numpy(np.real(H))
    H_torch[(i*BATCH_SIZE):((i+1)*BATCH_SIZE),:,:,N_t:2*N_t] = torch.from_numpy(np.imag(H))
#torch.save(H_torch,'data/H_train_UPA'+str(-1)+'Lp_1.pt')

H_torch = torch.zeros([BATCH_SIZE*N_BATCH_train,K,Nc,N_t*2])
for i in range(N_BATCH_train):
    H = np.zeros([BATCH_SIZE,K,Nc,N_t],dtype=complex) #D0: samples, D1: users, D2: subcarriers, D3: antennas
    for j in range(BATCH_SIZE):
        N = np.int(np.random.rand(1)*8)+1
        for k in range(K):
            H_f,A_t,alpha = freqency_sparse_SV_channel0(Nc,N,sigma_2_alpha,Nt)
            H[j,k,:,:] = H_f
    
    H_torch[(i*BATCH_SIZE):((i+1)*BATCH_SIZE),:,:,0:N_t] = torch.from_numpy(np.real(H))
    H_torch[(i*BATCH_SIZE):((i+1)*BATCH_SIZE),:,:,N_t:2*N_t] = torch.from_numpy(np.imag(H))

#torch.save(H_torch,'data/H_train_UPA'+str(-1)+'Lp_2.pt')

AttributeError: module 'numpy' has no attribute 'int'.
`np.int` was a deprecated alias for the builtin `int`. To avoid this error in existing code, use `int` by itself. Doing this will not modify any behavior and is safe. When replacing `np.int`, you may wish to use e.g. `np.int64` or `np.int32` to specify the precision. If you wish to review your current use, check the release note link for additional information.
The aliases was originally deprecated in NumPy 1.20; for more details and guidance see the original release note at:
    https://numpy.org/devdocs/release/1.20.0-notes.html#deprecations

In [14]:
H_torch = torch.zeros([BATCH_SIZE*N_BATCH_test,K,Nc,N_t*2])
for i in range(N_BATCH_test):
    H = np.zeros([BATCH_SIZE,K,Nc,N_t],dtype=complex) #第0个维度是样本 第1个维度是用户，第2个维度是子载波，第3个维度是天线
    for j in range(BATCH_SIZE):
        N = np.int(np.random.rand(1)*8)+1
        for k in range(K):
            H_f,A_t,alpha = freqency_sparse_SV_channel0(Nc,N,sigma_2_alpha,Nt)
            H[j,k,:,:] = H_f
    
    H_torch[(i*BATCH_SIZE):((i+1)*BATCH_SIZE),:,:,0:N_t] = torch.from_numpy(np.real(H))
    H_torch[(i*BATCH_SIZE):((i+1)*BATCH_SIZE),:,:,N_t:2*N_t] = torch.from_numpy(np.imag(H))
#     print(i)
# torch.save(H_torch,'data/H_test_UPA'+str(-1)+'Lp.pt')

AttributeError: module 'numpy' has no attribute 'int'.
`np.int` was a deprecated alias for the builtin `int`. To avoid this error in existing code, use `int` by itself. Doing this will not modify any behavior and is safe. When replacing `np.int`, you may wish to use e.g. `np.int64` or `np.int32` to specify the precision. If you wish to review your current use, check the release note link for additional information.
The aliases was originally deprecated in NumPy 1.20; for more details and guidance see the original release note at:
    https://numpy.org/devdocs/release/1.20.0-notes.html#deprecations

In [8]:
import scipy.io as io

H = np.zeros([BATCH_SIZE,K,Nc,N_t],dtype=complex) ##D0: samples, D1: users, D2: subcarriers, D3: antennas
for j in range(BATCH_SIZE):
    for k in range(K):
        H_f,A_t,alpha = freqency_sparse_SV_channel0(Nc,N,sigma_2_alpha,Nt)
        H[j,k,:,:] = H_f
H_torch = torch.zeros([BATCH_SIZE,K,Nc,N_t*2])
H_torch[:,:,:,0:N_t] = torch.from_numpy(np.real(H))
H_torch[:,:,:,N_t:2*N_t] = torch.from_numpy(np.imag(H))
#torch.save(H_torch,'data/H_test_UPA.pt')
print(Nt)


#dataNew = 'D:\python_project\model_quan\data\H_UPA.mat'
#io.savemat(dataNew, {'H_UPA':H})

[8, 8]
